# 1. Preparação Geral dos Dados e Constantes

Nesta etapa, carregamos a base limpa do NHIS 2025 (`adult25_limpo.csv`), definimos o `ID_DOMICILIO` como índice e isolamos a variável alvo multiclasse (`SEVERIDADE_DEPRESSAO_PHQ`), filtrando os registros indefinidos na Target (valores `-1` e `5`).

A coluna `PESO_AMOSTRAL` não é usada para ponderar o treinamento nem as métricas e é removida de $X$, junto com as colunas de sintomas diretos de depressão, para evitar vazamento de dados (*data leakage*). Os resultados descrevem apenas os respondentes analisados. O `class_weight='balanced'` continua no modelo para lidar com o desbalanceamento entre classes, sem extrapolar para a população. A linha `weighted avg` do relatório também é uma média ponderada pelo suporte das classes observadas, não pelo peso amostral.

In [38]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==============================================================================
# CONFIGURAÇÃO DE VARIÁVEIS GLOBAIS / CONSTANTES
# ==============================================================================
ID_COLUNA = 'ID_DOMICILIO'
PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET = 'SEVERIDADE_DEPRESSAO_PHQ'

COLUNAS_SINTOMAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM',
    'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM',
    'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM',
    'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM',
    'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO',
    'REMEDIO_DEPRESSAO',
    'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO'
]

# 1. Carregar a base limpa e definir o ID como índice
df = pd.read_csv('adult25_limpo.csv').set_index(ID_COLUNA)

# 2. Eliminar os registros indefinidos no TARGET (-1 e 5)
df = df[~df[TARGET].isin([-1, 5])].copy()

# 3. Isolar Target (y) e Features Preditoras (X), sem usar o peso amostral
y = df[TARGET]
colunas_remover = [TARGET, PESO_COLUNA] + COLUNAS_SINTOMAS_DEPRESSAO
X = df.drop(columns=[col for col in colunas_remover if col in df.columns])

# 2. Cenário A: Treinamento Mantendo Códigos Indeterminados (-1) nas Features

Neste primeiro cenário, mantemos as categorias de não-resposta (`-1`) presentes nas variáveis preditoras ($X$), permitindo que o modelo de árvore e a codificação One-Hot tratem o valor como uma categoria válida.

In [39]:
# Split Estratificado
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [col for col in X_train_A.columns if col not in num_cols]

preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_A = preprocessor_A.fit_transform(X_train_A)
X_test_proc_A = preprocessor_A.transform(X_test_A)

# Modelo Random Forest
model_A = RandomForestClassifier(random_state=42, class_weight='balanced')
model_A.fit(X_train_proc_A, y_train_A)

# Avaliação Cenário A
y_pred_A = model_A.predict(X_test_proc_A)
y_proba_A = model_A.predict_proba(X_test_proc_A)

print("--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---")
print(classification_report(y_test_A, y_pred_A, zero_division=0))

print("--- CENÁRIO A: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_A, y_pred_A)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_A, y_proba_A, multi_class='ovr'):.4f}")

--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.93      0.89      0.91      3721
           2       0.41      0.57      0.48       670
           3       0.32      0.18      0.23       217
           4       0.53      0.53      0.53       116

    accuracy                           0.80      4724
   macro avg       0.55      0.54      0.54      4724
weighted avg       0.82      0.80      0.81      4724

--- CENÁRIO A: Matriz de Confusão 4x4 ---
[[3300  405   14    2]
 [ 233  380   45   12]
 [  19  118   40   40]
 [   3   26   26   61]]
ROC-AUC Score (OvR): 0.9071


# 3. Cenário B: Treinamento Apenas com Informações Completas (Remoção de -1 no X)

Conforme discutido com a equipe e orientação do prof. Roy, realizamos um teste estrito filtrando qualquer observação que contenha não-resposta (`-1`) em qualquer feature preditora. A amostragem retida foi de **22.084 registros (93.5% dos dados)**.

In [40]:
# Filtragem Estrita das Features
mascara_sem_menos1 = (X != -1).all(axis=1)
X_strict = X[mascara_sem_menos1].copy()
y_strict = y[mascara_sem_menos1].copy()

print(f"Total de registros na base completa estrita: {len(X_strict)}")

# Split Estratificado Cenário B
X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
    X_strict, y_strict, test_size=0.2, random_state=42, stratify=y_strict
)

# Pré-processamento
preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_B = preprocessor_B.fit_transform(X_train_B)
X_test_proc_B = preprocessor_B.transform(X_test_B)

# Modelo Random Forest Cenário B
model_B = RandomForestClassifier(random_state=42, class_weight='balanced')
model_B.fit(X_train_proc_B, y_train_B)

# Avaliação Cenário B
y_pred_B = model_B.predict(X_test_proc_B)
y_proba_B = model_B.predict_proba(X_test_proc_B)

print("--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---")
print(classification_report(y_test_B, y_pred_B, zero_division=0))

print("--- CENÁRIO B: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_B, y_pred_B)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_B, y_proba_B, multi_class='ovr'):.4f}")

Total de registros na base completa estrita: 22084
--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.93      0.89      0.91      3482
           2       0.42      0.59      0.49       628
           3       0.37      0.22      0.27       202
           4       0.61      0.50      0.55       105

    accuracy                           0.80      4417
   macro avg       0.58      0.55      0.55      4417
weighted avg       0.82      0.80      0.81      4417

--- CENÁRIO B: Matriz de Confusão 4x4 ---
[[3084  385   11    2]
 [ 213  373   34    8]
 [  18  117   44   23]
 [   1   22   30   52]]
ROC-AUC Score (OvR): 0.9120


# 4. Otimização de Hiperparâmetros no Random Forest (RandomizedSearchCV)

Para buscar um melhor equilíbrio entre as classes de severidade e reduzir a sobreposição entre as classes Leve (2) e Moderada (3), aplicamos o `RandomizedSearchCV`. 

Buscamos otimizar a profundidade máxima das árvores (`max_depth`), o número de estimadores (`n_estimators`), e o tamanho mínimo de amostras em nós/folhas (`min_samples_split` e `min_samples_leaf`) utilizando validação cruzada estratificada em 3 folds.

In [41]:
from sklearn.model_selection import RandomizedSearchCV


# 1. Definir o Espaço de Busca de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.3],
    'class_weight': ['balanced', 'balanced_subsample']
}

# 2. Inicializar o Modelo Base
rf_base = RandomForestClassifier(random_state=42)

# 3. Configurar a Busca Aleatória (3 folds de validação cruzada)
random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=15,                   # Testa 15 combinações aleatórias
    scoring='f1_macro',          # Foca em melhorar a média macro entre as 4 classes
    cv=3,
    random_state=42,
    n_jobs=-1,                   # Usa todos os núcleos do processador
    verbose=1
)

# 4. Executar a Busca no Cenário B (Base Estritamente Limpa)
random_search.fit(X_train_proc_B, y_train_B)

print("\n--- Melhores Hiperparâmetros Encontrados ---")
print(random_search.best_params_)

# 5. Avaliar o Melhor Modelo no Conjunto de Teste
best_model = random_search.best_estimator_

y_pred_tuned = best_model.predict(X_test_proc_B)
y_proba_tuned = best_model.predict_proba(X_test_proc_B)

print("\n--- RELATÓRIO APÓS TUNING (Cenário B - 4 Classes) ---")
print(classification_report(y_test_B, y_pred_tuned, zero_division=0))

print("--- MATRIZ DE CONFUSÃO APÓS TUNING (4x4) ---")
print(np.round(confusion_matrix(y_test_B, y_pred_tuned)).astype(int))

print(f"\nROC-AUC Score (OvR) Otimizado: {roc_auc_score(y_test_B, y_proba_tuned, multi_class='ovr'):.4f}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits

--- Melhores Hiperparâmetros Encontrados ---
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 15, 'class_weight': 'balanced'}

--- RELATÓRIO APÓS TUNING (Cenário B - 4 Classes) ---
              precision    recall  f1-score   support

           1       0.95      0.85      0.89      3482
           2       0.39      0.63      0.48       628
           3       0.38      0.36      0.37       202
           4       0.58      0.67      0.62       105

    accuracy                           0.79      4417
   macro avg       0.58      0.62      0.59      4417
weighted avg       0.84      0.79      0.81      4417

--- MATRIZ DE CONFUSÃO APÓS TUNING (4x4) ---
[[2943  511   25    3]
 [ 146  396   69   17]
 [  10   89   72   31]
 [   0   11   24   70]]

ROC-AUC Score (OvR) Otimizado: 0.9072


In [42]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Carregar a base limpa
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')

# 2. Remover indeterminados (-1 e 5) da Target PHQ original
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()

# 3. Aplicar a transformação binária (Sugestão do Alexandre)
# 0 = Sem Depressão (Nível 1) | 1 = Com Depressão (Níveis 2, 3 e 4)
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ' # Removemos a target original do X
]

# 4. Isolar X e y e filtrar registros completos (sem -1 nas preditoras)
y_bin = df[TARGET_BIN]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

# Filtro de registros completos (22.084 linhas)
mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()

# 5. Split Estratificado Binário
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 6. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA', 'IDADE']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)

# 7. Treinar Random Forest Binário (com hiperparâmetros otimizados)
rf_binario = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    max_features='log2',
    class_weight='balanced',
    random_state=42
)
rf_binario.fit(X_tr_proc, y_tr)

# 8. Avaliação do Modelo Binário
y_pred = rf_binario.predict(X_te_proc)
y_proba = rf_binario.predict_proba(X_te_proc)[:, 1]

print("--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---")
print(classification_report(y_te, y_pred, zero_division=0))

print("--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---")
print(np.round(confusion_matrix(y_te, y_pred)).astype(int))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba):.4f}")

--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---
              precision    recall  f1-score   support

           0       0.93      0.90      0.91      3482
           1       0.66      0.76      0.70       935

    accuracy                           0.87      4417
   macro avg       0.80      0.83      0.81      4417
weighted avg       0.87      0.87      0.87      4417

--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---
[[3118  364]
 [ 229  706]]

ROC-AUC Score (Binário): 0.9137


In [43]:
# 1. Carregar a base limpa
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')

# 2. Remover indeterminados (-1 e 5) da Target PHQ original
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()

# 3. Aplicar a transformação binária (Sugestão do Alexandre)
# 0 = Sem Depressão (Nível 1) | 1 = Com Depressão (Níveis 2, 3 e 4)
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ','FREQ_ANSIEDADE_2SEM',
      'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
      'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
      'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM', 'FREQ_MEDO_2SEM',
    'SEVERIDADE_ANSIEDADE', 'HISTORICO_TRANSTORNO_ANSIEDADE', 
    'MED_ANSIEDADE', 'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE' # Removemos as variáveis relacionadas à saude mental
]

# 4. Isolar X e y e filtrar registros completos (sem -1 nas preditoras)
y_bin = df[TARGET_BIN]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

# Filtro de registros completos (22.084 linhas)
mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()

# 5. Split Estratificado Binário
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 6. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA', 'IDADE']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)

# 7. Treinar Random Forest Binário (com hiperparâmetros otimizados)
rf_binario = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    max_features='log2',
    class_weight='balanced',
    random_state=42
)
rf_binario.fit(X_tr_proc, y_tr)

# Variáveis preditoras originais e importância das variáveis após o One-Hot Encoding
print(f"Variáveis preditoras originais utilizadas ({len(X_tr.columns)}):")
print("\n".join(f"- {coluna}" for coluna in X_tr.columns))

importancias = pd.DataFrame(
    {
        "variavel_transformada": preprocessor.get_feature_names_out(),
        "importancia": rf_binario.feature_importances_,
    }
).sort_values("importancia", ascending=False)
print("\nVariáveis transformadas mais importantes para o modelo:")
display(importancias.head(20))

# 8. Avaliação do Modelo Binário
y_pred = rf_binario.predict(X_te_proc)
y_proba = rf_binario.predict_proba(X_te_proc)[:, 1]

print("--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---")
print(classification_report(y_te, y_pred, zero_division=0))

print("--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---")
print(np.round(confusion_matrix(y_te, y_pred)).astype(int))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba):.4f}")

Variáveis preditoras originais utilizadas (46):
- IDADE
- SEXO
- CLASSIFICACAO_URBANO_RURAL
- REGIAO_GEOGRAFICA
- ESTADO_CIVIL_DECLARADO
- ESTADO_CIVIL
- STATUS_PARENTAL
- SEXO_CONJUGE
- SEPARACAO_LEGAL_CONJUGAL
- COJUGE_TRABALHA
- FLG_IDOSO_FAMILIA
- QTD_ADULTOS_FAMILIA
- QTD_CRIANÇAS_FAMILIA
- ESCOLARIDADE_MAXIMA_FAMILIA
- TEMPO_MORADIA_ANOS
- TIPO_POSSE_IMOVEL
- SUPORTE_SOCIAL
- NATURAL_ESTADOS_UNIDOS
- AUTOAVALIACAO_SAUDE_GERAL
- CATEGORIA_IMC
- HISTORICO_COLESTEROL
- HISTORICO_DIABETES
- HISTORICO_HIPERTENSAO
- HISTORICO_ARTRITE
- CONCUSSAO_SINTOMAS_POS_TRAUMA
- DIFICULDADE_LOCOMOCAO
- LIMITACAO_TRABALHO_POR_SAUDE
- DEFICIENTE
- ASMA_ATIVA_ATUALMENTE
- DOR_CRONICA
- TEVE_CANCER
- ULTIMA_VISITA_MEDICO
- VISITAS_PRONTO_ATENDIMENTO_12M
- VISITAS_EMERGENCIA_HOSPITALAR_12M
- VISITAS_EXAME_VISTA_12M
- CATEGORIA_RAZAO_POBREZA
- RAZAO_RENDA_POBREZA
- RENDA_INVESTIMENTOS
- PLANO_PUBLICO
- PLANO_PRIVADO
- ESTRESSE_FINANCEIRO_MEDICO
- JA_FUMANTE
- FREQ_SOLITARIO
- FREQUENCIA_FALHA_MEMORIA
- 

,variavel_transformada,importancia
123,cat__TIPO_DIFICULDADE_COGNITIVA_4,0.047370
1,num__IDADE,0.046275
0,num__RAZAO_RENDA_POBREZA,0.040561
115,cat__FREQ_SOLITARIO_3,0.038915
120,cat__FREQUENCIA_FALHA_MEMORIA_4,0.031815
54,cat__SUPORTE_SOCIAL_5,0.030774
124,cat__QTD_DIFICULDADE_COGNITIVA_2,0.028831
122,cat__TIPO_DIFICULDADE_COGNITIVA_3,0.026604
72,cat__LIMITACAO_TRABALHO_POR_SAUDE_2,0.025277
77,cat__DOR_CRONICA_4,0.023395


--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      3502
           1       0.60      0.66      0.62       945

    accuracy                           0.83      4447
   macro avg       0.75      0.77      0.76      4447
weighted avg       0.84      0.83      0.84      4447

--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---
[[3085  417]
 [ 326  619]]

ROC-AUC Score (Binário): 0.8639


In [44]:
# 1. Carregar a base limpa
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')

# 2. Remover indeterminados (-1 e 5) da Target PHQ original
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()

# 3. Aplicar a transformação binária (Sugestão do Alexandre)
# 0 = Sem Depressão (Nível 1) | 1 = Com Depressão (Níveis 2, 3 e 4)
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ','FREQ_ANSIEDADE_2SEM',
      'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
      'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
      'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM', 'FREQ_MEDO_2SEM',
    'SEVERIDADE_ANSIEDADE', 'HISTORICO_TRANSTORNO_ANSIEDADE', 
    'MED_ANSIEDADE', 'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE' # Removemos as variáveis relacionadas à saude mental
]

# 4. Isolar X e y e filtrar registros completos (sem -1 nas preditoras)
y_bin = df[TARGET_BIN]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

# Filtro de registros completos (22.084 linhas)
mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()

# 5. Split Estratificado Binário
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 6. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA', 'IDADE']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)

# 7. Treinar Random Forest Binário (com hiperparâmetros otimizados)
rf_binario = RandomForestClassifier(
    n_estimators=300,
    max_depth= None,
    min_samples_split=5,
    max_features='log2',
    class_weight='balanced',
    random_state=42
)
rf_binario.fit(X_tr_proc, y_tr)

# Variáveis preditoras originais e importância das variáveis após o One-Hot Encoding
print(f"Variáveis preditoras originais utilizadas ({len(X_tr.columns)}):")
print("\n".join(f"- {coluna}" for coluna in X_tr.columns))

importancias = pd.DataFrame(
    {
        "variavel_transformada": preprocessor.get_feature_names_out(),
        "importancia": rf_binario.feature_importances_,
    }
).sort_values("importancia", ascending=False)
print("\nVariáveis transformadas mais importantes para o modelo:")
display(importancias.head(20))

# 8. Avaliação do Modelo Binário
y_pred = rf_binario.predict(X_te_proc)
y_proba = rf_binario.predict_proba(X_te_proc)[:, 1]

print("--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---")
print(classification_report(y_te, y_pred, zero_division=0))

print("--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---")
print(np.round(confusion_matrix(y_te, y_pred)).astype(int))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba):.4f}")

Variáveis preditoras originais utilizadas (46):
- IDADE
- SEXO
- CLASSIFICACAO_URBANO_RURAL
- REGIAO_GEOGRAFICA
- ESTADO_CIVIL_DECLARADO
- ESTADO_CIVIL
- STATUS_PARENTAL
- SEXO_CONJUGE
- SEPARACAO_LEGAL_CONJUGAL
- COJUGE_TRABALHA
- FLG_IDOSO_FAMILIA
- QTD_ADULTOS_FAMILIA
- QTD_CRIANÇAS_FAMILIA
- ESCOLARIDADE_MAXIMA_FAMILIA
- TEMPO_MORADIA_ANOS
- TIPO_POSSE_IMOVEL
- SUPORTE_SOCIAL
- NATURAL_ESTADOS_UNIDOS
- AUTOAVALIACAO_SAUDE_GERAL
- CATEGORIA_IMC
- HISTORICO_COLESTEROL
- HISTORICO_DIABETES
- HISTORICO_HIPERTENSAO
- HISTORICO_ARTRITE
- CONCUSSAO_SINTOMAS_POS_TRAUMA
- DIFICULDADE_LOCOMOCAO
- LIMITACAO_TRABALHO_POR_SAUDE
- DEFICIENTE
- ASMA_ATIVA_ATUALMENTE
- DOR_CRONICA
- TEVE_CANCER
- ULTIMA_VISITA_MEDICO
- VISITAS_PRONTO_ATENDIMENTO_12M
- VISITAS_EMERGENCIA_HOSPITALAR_12M
- VISITAS_EXAME_VISTA_12M
- CATEGORIA_RAZAO_POBREZA
- RAZAO_RENDA_POBREZA
- RENDA_INVESTIMENTOS
- PLANO_PUBLICO
- PLANO_PRIVADO
- ESTRESSE_FINANCEIRO_MEDICO
- JA_FUMANTE
- FREQ_SOLITARIO
- FREQUENCIA_FALHA_MEMORIA
- 

,variavel_transformada,importancia
1,num__IDADE,0.046441
123,cat__TIPO_DIFICULDADE_COGNITIVA_4,0.044143
0,num__RAZAO_RENDA_POBREZA,0.041317
115,cat__FREQ_SOLITARIO_3,0.039293
120,cat__FREQUENCIA_FALHA_MEMORIA_4,0.031203
54,cat__SUPORTE_SOCIAL_5,0.029959
124,cat__QTD_DIFICULDADE_COGNITIVA_2,0.028430
122,cat__TIPO_DIFICULDADE_COGNITIVA_3,0.026910
72,cat__LIMITACAO_TRABALHO_POR_SAUDE_2,0.025034
77,cat__DOR_CRONICA_4,0.022916


--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---
              precision    recall  f1-score   support

           0       0.90      0.89      0.90      3502
           1       0.61      0.65      0.63       945

    accuracy                           0.84      4447
   macro avg       0.76      0.77      0.76      4447
weighted avg       0.84      0.84      0.84      4447

--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---
[[3104  398]
 [ 330  615]]

ROC-AUC Score (Binário): 0.8640


In [45]:
# 1. Carregar a base limpa
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')

# 2. Remover indeterminados (-1 e 5) da Target PHQ original
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()

# 3. Aplicar a transformação binária (Sugestão do Alexandre)
# 0 = Sem Depressão (Nível 1) | 1 = Com Depressão (Níveis 2, 3 e 4)
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ','FREQ_ANSIEDADE_2SEM',
      'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
      'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
      'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM', 'FREQ_MEDO_2SEM',
    'SEVERIDADE_ANSIEDADE', 'HISTORICO_TRANSTORNO_ANSIEDADE', 
    'MED_ANSIEDADE', 'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE' # Removemos as variáveis relacionadas à saude mental
]

# 4. Isolar X e y e filtrar registros completos (sem -1 nas preditoras)
y_bin = df[TARGET_BIN]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

# Filtro de registros completos (22.084 linhas)
mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()

# 5. Split Estratificado Binário
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 6. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA', 'IDADE']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)

# 7. Treinar Random Forest Binário (com hiperparâmetros otimizados)
rf_binario = RandomForestClassifier(
    n_estimators=300,
    max_depth= None,
    min_samples_split=2,
    max_features='log2',
    class_weight='balanced',
    random_state=42
)
rf_binario.fit(X_tr_proc, y_tr)

# Variáveis preditoras originais e importância das variáveis após o One-Hot Encoding
print(f"Variáveis preditoras originais utilizadas ({len(X_tr.columns)}):")
print("\n".join(f"- {coluna}" for coluna in X_tr.columns))

importancias = pd.DataFrame(
    {
        "variavel_transformada": preprocessor.get_feature_names_out(),
        "importancia": rf_binario.feature_importances_,
    }
).sort_values("importancia", ascending=False)
print("\nVariáveis transformadas mais importantes para o modelo:")
display(importancias.head(20))

# 8. Avaliação do Modelo Binário
y_pred = rf_binario.predict(X_te_proc)
y_proba = rf_binario.predict_proba(X_te_proc)[:, 1]

print("--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---")
print(classification_report(y_te, y_pred, zero_division=0))

print("--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---")
print(np.round(confusion_matrix(y_te, y_pred)).astype(int))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba):.4f}")

Variáveis preditoras originais utilizadas (46):
- IDADE
- SEXO
- CLASSIFICACAO_URBANO_RURAL
- REGIAO_GEOGRAFICA
- ESTADO_CIVIL_DECLARADO
- ESTADO_CIVIL
- STATUS_PARENTAL
- SEXO_CONJUGE
- SEPARACAO_LEGAL_CONJUGAL
- COJUGE_TRABALHA
- FLG_IDOSO_FAMILIA
- QTD_ADULTOS_FAMILIA
- QTD_CRIANÇAS_FAMILIA
- ESCOLARIDADE_MAXIMA_FAMILIA
- TEMPO_MORADIA_ANOS
- TIPO_POSSE_IMOVEL
- SUPORTE_SOCIAL
- NATURAL_ESTADOS_UNIDOS
- AUTOAVALIACAO_SAUDE_GERAL
- CATEGORIA_IMC
- HISTORICO_COLESTEROL
- HISTORICO_DIABETES
- HISTORICO_HIPERTENSAO
- HISTORICO_ARTRITE
- CONCUSSAO_SINTOMAS_POS_TRAUMA
- DIFICULDADE_LOCOMOCAO
- LIMITACAO_TRABALHO_POR_SAUDE
- DEFICIENTE
- ASMA_ATIVA_ATUALMENTE
- DOR_CRONICA
- TEVE_CANCER
- ULTIMA_VISITA_MEDICO
- VISITAS_PRONTO_ATENDIMENTO_12M
- VISITAS_EMERGENCIA_HOSPITALAR_12M
- VISITAS_EXAME_VISTA_12M
- CATEGORIA_RAZAO_POBREZA
- RAZAO_RENDA_POBREZA
- RENDA_INVESTIMENTOS
- PLANO_PUBLICO
- PLANO_PRIVADO
- ESTRESSE_FINANCEIRO_MEDICO
- JA_FUMANTE
- FREQ_SOLITARIO
- FREQUENCIA_FALHA_MEMORIA
- 

,variavel_transformada,importancia
1,num__IDADE,0.047484
0,num__RAZAO_RENDA_POBREZA,0.044362
123,cat__TIPO_DIFICULDADE_COGNITIVA_4,0.034484
115,cat__FREQ_SOLITARIO_3,0.033153
120,cat__FREQUENCIA_FALHA_MEMORIA_4,0.028385
54,cat__SUPORTE_SOCIAL_5,0.026155
124,cat__QTD_DIFICULDADE_COGNITIVA_2,0.023077
122,cat__TIPO_DIFICULDADE_COGNITIVA_3,0.021315
72,cat__LIMITACAO_TRABALHO_POR_SAUDE_2,0.020486
77,cat__DOR_CRONICA_4,0.019547


--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---
              precision    recall  f1-score   support

           0       0.89      0.91      0.90      3502
           1       0.64      0.58      0.61       945

    accuracy                           0.84      4447
   macro avg       0.76      0.74      0.75      4447
weighted avg       0.83      0.84      0.84      4447

--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---
[[3189  313]
 [ 399  546]]

ROC-AUC Score (Binário): 0.8632


In [46]:
# 1. Carregar a base limpa
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')

# 2. Remover indeterminados (-1 e 5) da Target PHQ original
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()

# 3. Aplicar a transformação binária (Sugestão do Alexandre)
# 0 = Sem Depressão (Nível 1) | 1 = Com Depressão (Níveis 2, 3 e 4)
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ','FREQ_ANSIEDADE_2SEM',
      'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
      'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
      'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM', 'FREQ_MEDO_2SEM',
    'SEVERIDADE_ANSIEDADE', 'HISTORICO_TRANSTORNO_ANSIEDADE', 
    'MED_ANSIEDADE', 'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE' # Removemos as variáveis relacionadas à saude mental
]

# 4. Isolar X e y e filtrar registros completos (sem -1 nas preditoras)
y_bin = df[TARGET_BIN]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

# Filtro de registros completos (22.084 linhas)
mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()

# 5. Split Estratificado Binário
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 6. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA', 'IDADE']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)

# 7. Treinar Random Forest Binário (com hiperparâmetros otimizados)
rf_binario = RandomForestClassifier(
    n_estimators=300,
    max_depth= None,
    min_samples_split=2,
    max_features=None,
    class_weight='balanced',
    random_state=42
)
rf_binario.fit(X_tr_proc, y_tr)

# Variáveis preditoras originais e importância das variáveis após o One-Hot Encoding
print(f"Variáveis preditoras originais utilizadas ({len(X_tr.columns)}):")
print("\n".join(f"- {coluna}" for coluna in X_tr.columns))

importancias = pd.DataFrame(
    {
        "variavel_transformada": preprocessor.get_feature_names_out(),
        "importancia": rf_binario.feature_importances_,
    }
).sort_values("importancia", ascending=False)
print("\nVariáveis transformadas mais importantes para o modelo:")
display(importancias.head(20))

# 8. Avaliação do Modelo Binário
y_pred = rf_binario.predict(X_te_proc)
y_proba = rf_binario.predict_proba(X_te_proc)[:, 1]

print("--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---")
print(classification_report(y_te, y_pred, zero_division=0))

print("--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---")
print(np.round(confusion_matrix(y_te, y_pred)).astype(int))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba):.4f}")

Variáveis preditoras originais utilizadas (46):
- IDADE
- SEXO
- CLASSIFICACAO_URBANO_RURAL
- REGIAO_GEOGRAFICA
- ESTADO_CIVIL_DECLARADO
- ESTADO_CIVIL
- STATUS_PARENTAL
- SEXO_CONJUGE
- SEPARACAO_LEGAL_CONJUGAL
- COJUGE_TRABALHA
- FLG_IDOSO_FAMILIA
- QTD_ADULTOS_FAMILIA
- QTD_CRIANÇAS_FAMILIA
- ESCOLARIDADE_MAXIMA_FAMILIA
- TEMPO_MORADIA_ANOS
- TIPO_POSSE_IMOVEL
- SUPORTE_SOCIAL
- NATURAL_ESTADOS_UNIDOS
- AUTOAVALIACAO_SAUDE_GERAL
- CATEGORIA_IMC
- HISTORICO_COLESTEROL
- HISTORICO_DIABETES
- HISTORICO_HIPERTENSAO
- HISTORICO_ARTRITE
- CONCUSSAO_SINTOMAS_POS_TRAUMA
- DIFICULDADE_LOCOMOCAO
- LIMITACAO_TRABALHO_POR_SAUDE
- DEFICIENTE
- ASMA_ATIVA_ATUALMENTE
- DOR_CRONICA
- TEVE_CANCER
- ULTIMA_VISITA_MEDICO
- VISITAS_PRONTO_ATENDIMENTO_12M
- VISITAS_EMERGENCIA_HOSPITALAR_12M
- VISITAS_EXAME_VISTA_12M
- CATEGORIA_RAZAO_POBREZA
- RAZAO_RENDA_POBREZA
- RENDA_INVESTIMENTOS
- PLANO_PUBLICO
- PLANO_PRIVADO
- ESTRESSE_FINANCEIRO_MEDICO
- JA_FUMANTE
- FREQ_SOLITARIO
- FREQUENCIA_FALHA_MEMORIA
- 

,variavel_transformada,importancia
123,cat__TIPO_DIFICULDADE_COGNITIVA_4,0.153352
1,num__IDADE,0.079699
0,num__RAZAO_RENDA_POBREZA,0.068248
115,cat__FREQ_SOLITARIO_3,0.052486
77,cat__DOR_CRONICA_4,0.026259
116,cat__FREQ_SOLITARIO_4,0.024749
54,cat__SUPORTE_SOCIAL_5,0.018905
72,cat__LIMITACAO_TRABALHO_POR_SAUDE_2,0.018721
117,cat__FREQ_SOLITARIO_5,0.013588
112,cat__ESTRESSE_FINANCEIRO_MEDICO_3,0.012573


--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---
              precision    recall  f1-score   support

           0       0.90      0.89      0.89      3502
           1       0.60      0.62      0.61       945

    accuracy                           0.83      4447
   macro avg       0.75      0.75      0.75      4447
weighted avg       0.83      0.83      0.83      4447

--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---
[[3110  392]
 [ 359  586]]

ROC-AUC Score (Binário): 0.8551
